# S4–S5 · Regression arms at a strictly matched budget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s4_regression_arms.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s4_regression_arms.ipynb)

Weeks 4–5 compare six regression arms that share the backbone (2.5D U-Net), the data, the folds, the number of steps and
the seeds; only the objective changes.

| Arm | Objective | Role |
|---|---|---|
| R1 | L1 in HU | current practice (Bayes act: median) |
| R2 | L2 in HU | control (mean in HU) |
| R3 | L2 in SPR | Bayes act for squared range error (docs/theory.md, Proposition 4) |
| R4 | R3 + per-tissue SPR offset, cross-fitted | mandatory cheap control |
| R5 | R3 + λ · mean ΔWET² over plan beams and random directions | method |
| R6 | R5 + β · FZ0 score of a per-ray (VaR, ES) head | method, tails |

The question and the decision rule are fixed in `configs/s4_preregistration.json` before real predictions exist:
does R5 or R6 lower the CVaR90 of the range error to the distal target edge relative to R3 at an MAE at most 5 % above R3's, and
does it beat the bias-corrected R4? If only R3 is beaten, the gain is a tissue-level bias and *the right estimator
suffices*, which is publishable on its own (plan, S6). The verdict uses the range proxy; the γ-proton part of go/no-go 2
needs the S6 dosimetry subset.

Everything runs from this notebook, locally or on Colab: download, cache, training of R1–R3, R5 and R6 for every
seed (runs under `outputs/s4/runs/seed<k>/<arm>`), the R4 correction, evaluation and the verdict. The full budget is
5 arms × 5 folds × 3 seeds; every fold is checkpointed, so it can be spread over several Colab sessions. Without a GPU,
training is skipped and existing runs are evaluated; with none, the notebook stops with a clear message.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead). `SCT_TASKS` selects the tasks (default `1,2`).

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables
of the same meaning, given in brackets, override them for unattended runs (`jupyter nbconvert --execute`).

Training runs here when a GPU is available (Colab: *Runtime → Change runtime type → GPU*). Each finished fold is
checkpointed and reused, so an interrupted session resumes where it stopped; set `TRAIN = False` to evaluate existing
runs only.
The pre-registered verdict assumes the defaults (30 000 steps, 5 folds, seeds 0,1,2); smaller values give a quick pass only.
| Platform | Dataset, cache | Outputs | GPU |
|---|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` | every visible GPU |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) | *Runtime → Change runtime type → T4 GPU* |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` | *Settings → Accelerator → GPU T4 × 2*, *Internet on* |

With two GPUs (Kaggle T4 × 2) the remaining folds train in parallel, one per GPU. T4 GPUs train in float16 with loss
scaling (they have no native bfloat16). Predictions are stored as int16 HU to fit the 20 GB of `/kaggle/working`.

In [ ]:
# @title Parameters { display-mode: "form" }
TASK = "1"  # @param ["1", "2"]  1 = MRI, 2 = CBCT [SCT_TASK]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
TRAIN = True  # @param {type:"boolean"}  train missing folds here (needs a GPU) [SCT_TRAIN_IN_NOTEBOOK]
STEPS = 30000  # @param {type:"integer"}  optimisation steps per fold [SCT_STEPS]
FOLDS = 5  # @param {type:"integer"}  cross-fitting folds [SCT_FOLDS]
SEEDS = "0,1,2"  # @param {type:"string"}  [SCT_SEEDS]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only

In [ ]:
import json
import os
import warnings
from dataclasses import asdict

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.correction import crossfit_bias_correction
from sctrange.data import discover_cases, load_case, read_volume
from sctrange.diagnostics import Preregistration, decide_arms, holm, paired_test
from sctrange.evaluation import DiagnosticConfig, evaluate_case, evaluate_risk
from sctrange.hlut import HLUT
from sctrange.plotting import apply_style, plot_arm_pareto, plot_training_curves
from sctrange.download import ensure_dataset
from sctrange.training import visible_devices
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_split, split_hash
from sctrange.training import ARMS, arm_config, crossfit

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
TRAIN = os.environ.get("SCT_TRAIN_IN_NOTEBOOK", "1" if TRAIN else "0") == "1"
STEPS = int(os.environ.get("SCT_STEPS", STEPS))
N_FOLDS = int(os.environ.get("SCT_FOLDS", FOLDS))
SEEDS = tuple(int(s) for s in os.environ.get("SCT_SEEDS", SEEDS).split(","))

OUTPUT_DIR = PATHS.output_root / "s4"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_RUNS_ROOT", OUTPUT_DIR / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", PATHS.cache_root / "cache"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s4_preregistration.json"

SEED = 20261013
DEVICES = visible_devices()  # ["cuda:0", "cuda:1"] on Kaggle T4 x 2
DEVICE = DEVICES[0]
TRAINED_ARMS = tuple(sorted(ARMS))
ARM_ORDER = ("R1", "R2", "R3", "R4", "R5", "R6")
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
MAX_CASES = None
OVERRIDES: dict = {"steps": STEPS}
hlut = HLUT.pyradplan_default()

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
TRAIN_IN_NOTEBOOK = TRAIN and (DEVICE.startswith("cuda") or os.environ.get("SCT_TRAIN_ON_CPU") == "1")
if TRAIN and not TRAIN_IN_NOTEBOOK:
    print("no GPU: training skipped, existing runs only (Colab: Runtime > Change runtime type > GPU; "
          "SCT_TRAIN_ON_CPU=1 forces CPU training)")
SPLIT_PATH = FROZEN_SPLIT_PATH

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
print(f"task {TASK} · devices {', '.join(DEVICES)} · folds {N_FOLDS} · steps {STEPS} · seeds {SEEDS} · "
      f"train in notebook: {TRAIN_IN_NOTEBOOK}")

## 1 · Frozen split and pre-registration

The frozen split of week 1 is verified, never regenerated on real data; test cases are never loaded. The
pre-registration file is hashed into the report.

In [ ]:
records = discover_cases(DATA_ROOT)
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(SPLIT_PATH)

development_ids = set(split["development"])
development = [r for r in records if r.case_id in development_ids and r.task == TASK][:MAX_CASES]
assert not development_ids & set(split["test"])

prereg = Preregistration.load(PREREGISTRATION_PATH)
display(pd.Series({"split sha256": split_hash(split), "development cases (this task)": len(development),
                   "test cases (never loaded)": len(split["test"]),
                   "pre-registration sha256": prereg.fingerprint}, name="value").to_frame())
display(pd.json_normalize({k: v for k, v in prereg.raw.items() if k != "verdicts"}, sep=" · ").T.rename(columns={0: "value"}))

## 2 · Training (cross-fitted) and the R4 control

For every seed, each trained arm is cross-fitted on the development pool; R4 is then derived from the out-of-fold R3
predictions, with offsets for fold *k* estimated on the other folds only. Checkpoints are reused when they match the
configuration and refused otherwise, so reruns are cheap and arms cannot be mixed up.

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")

histories, missing, offsets = {}, [], {}
for seed in SEEDS:
    seed_dir = RUNS_ROOT / f"seed{seed}"
    for arm in TRAINED_ARMS:
        run_dir = seed_dir / arm
        if TRAIN_IN_NOTEBOOK:
            crossfit(development, cache, arm_config(arm, seed=seed, device=DEVICE, **OVERRIDES), hlut, N_FOLDS, run_dir,
                     DEVICES)
        if not (run_dir / "predictions").exists():
            missing.append(f"seed{seed}/{arm}")
        elif (run_dir / "history.csv").exists() and seed == SEEDS[0]:
            histories[arm] = pd.read_csv(run_dir / "history.csv")
    if (seed_dir / "R3" / "folds.csv").exists():
        offsets[seed] = crossfit_bias_correction(development, seed_dir / "R3", seed_dir / "R4", hlut)
    else:
        missing.append(f"seed{seed}/R4 (needs R3)")

print("missing runs:", ", ".join(missing) if missing else "none")
display(pd.Series(asdict(arm_config("R6", seed=SEEDS[0], device=DEVICE, **OVERRIDES)), name="R6 configuration").to_frame())
if offsets:
    display(pd.concat(offsets, names=["seed"]).round(4).rename_axis(columns="R4 SPR offset by predicted class"))
if histories:
    plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()

## 3 · Per-case evaluation

Every out-of-fold prediction of every arm and seed is evaluated with the S1 metrics (8 coplanar beams, 25 mm surrogate
target at the body centroid). R6 risk maps are integrated along the same segments to check the calibration of the
predicted VaR90 and ES90. Per-case metrics are averaged over seeds before any test.

In [ ]:
rows, risk_rows = [], []
for record in development:
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    for seed in SEEDS:
        for arm in ARM_ORDER:
            run_dir = RUNS_ROOT / f"seed{seed}" / arm
            path = run_dir / "predictions" / f"{record.case_id}.mha"
            if not path.exists():
                continue
            sct = read_volume(path).array
            keys = {"case_id": record.case_id, "region": record.region, "center": record.center, "seed": seed, "source": arm}
            rows.append({**keys, **evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)})
            var_path = run_dir / "risk" / f"{record.case_id}_var.mha"
            if var_path.exists():
                var = read_volume(var_path).array
                excess = read_volume(run_dir / "risk" / f"{record.case_id}_excess.mha").array
                risk_rows.append({**keys, **evaluate_risk(ct, mask, sct, var, excess, spacing, hlut, DIAGNOSTIC)})

if not rows:
    raise RuntimeError(f"no out-of-fold prediction under {RUNS_ROOT}: run this notebook on a GPU with TRAIN = True "
                       "(Colab: Runtime > Change runtime type > GPU), or train with `sctrange-train --arm`")
per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUTPUT_DIR / "s4_per_seed.csv", index=False)
numeric = [c for c in per_seed.columns if c not in ("case_id", "region", "center", "seed", "source")]
diagnostics = per_seed.groupby(["case_id", "region", "center", "source"], as_index=False)[numeric].mean()
diagnostics.to_csv(OUTPUT_DIR / "s4_diagnostics.csv", index=False)
risk = pd.DataFrame(risk_rows)
print(f"{diagnostics['case_id'].nunique()} cases · arms {sorted(diagnostics['source'].unique())} · "
      f"{len(per_seed)} evaluations over {len(SEEDS)} seed(s)")

columns = ["mae_hu", "target_mean_signed_mm", "target_rms_mm", "target_p95_mm", "target_cvar_mm", "target_coherence",
           "target_global_ratio"]
order = [arm for arm in ARM_ORDER if arm in set(diagnostics["source"])]
display(diagnostics.groupby("source")[columns].median().reindex(order).round(3))

## 4 · Image error versus range-error tail

Figure 4 of the plan: one point per arm at the median over cases, whiskers spanning the interquartile range. Lower left
is better. The theory predicts that R3 improves on R1 mostly on the signed bias and the tail, not on MAE, and that any
gain of R5 or R6 shows on the vertical axis.

In [ ]:
grouped = diagnostics.groupby("source")
pareto = pd.concat({f"{metric}{suffix}": grouped[metric].quantile(q)
                    for metric in ("mae_hu", "target_cvar_mm")
                    for suffix, q in (("", 0.5), ("_q25", 0.25), ("_q75", 0.75))}, axis=1).reindex(order)
plot_arm_pareto(pareto).savefig(FIGURE_DIR / "fig4_pareto.png", bbox_inches="tight")
plt.show()
display(pareto.round(3))
display(diagnostics.pivot_table(index="region", columns="source", values="target_mean_signed_mm", aggfunc="median")
        .reindex(columns=order).round(3).rename_axis(columns="median signed ΔWET (mm)"))

## 5 · Pre-registered decision and secondary tests

The verdict applies `configs/s4_preregistration.json` unchanged. Secondary tests are reported whatever the verdict:
Proposition 5 on patients (signed abdominal range bias of R1 above R3, one-sided), and the MAE cost of each arm relative
to R3 (two-sided), all Holm-adjusted within this family.

In [ ]:
decision = decide_arms(diagnostics, prereg)
display(pd.Series({k: v for k, v in decision.items() if k not in ("arms", "raw_p_values")}, name="value").to_frame())
display(pd.DataFrame(decision["arms"]).T.round(4))

secondary = {"signed bias AB, R1 − R3 > 0": paired_test(diagnostics, "target_mean_signed_mm", "R1", "R3", "AB", "greater")}
for arm in ("R1", "R2", "R4", "R5", "R6"):
    secondary[f"MAE {arm} − R3"] = paired_test(diagnostics, "mae_hu", arm, "R3")
adjusted = holm({name: result["p_value"] for name, result in secondary.items()})
secondary_table = pd.DataFrame(secondary).T.assign(p_holm=pd.Series(adjusted))
display(secondary_table.round(4))

## 6 · Calibration of the R6 risk head

A calibrated head covers 90 % of rays with its VaR90 (`risk_coverage` ≈ 0.9), and the mean |ΔWET| beyond the predicted
VaR matches the predicted ES (`risk_realised_es_mm` ≈ `risk_mean_es_mm`). `risk_spearman` is the rank correlation
between predicted VaR and realised |ΔWET| over rays: the head is only useful for failure detection if it is positive.

In [ ]:
risk_columns = ["risk_rays", "risk_coverage", "risk_mean_var_mm", "risk_mean_es_mm", "risk_realised_es_mm", "risk_fz",
                "risk_spearman"]
if risk.empty:
    print("no risk maps found (R6 not trained yet)")
else:
    risk.to_csv(OUTPUT_DIR / "s4_risk.csv", index=False)
    calibration = risk.groupby("region")[risk_columns].median()
    calibration.loc["all"] = risk[risk_columns].median()
    display(calibration.round(3))

## 7 · Report

In [ ]:
report = {
    "task": TASK,
    "seeds": list(SEEDS),
    "folds": N_FOLDS,
    "split_sha256": split_hash(split),
    "overrides": OVERRIDES,
    "decision": decision,
    "secondary": secondary_table.to_dict(orient="index"),
    "medians_by_arm": diagnostics.groupby("source")[columns].median().to_dict(orient="index"),
    "risk_calibration": {} if risk.empty else risk[risk_columns].median().to_dict(),
    "missing_runs": missing,
}
(OUTPUT_DIR / "s4_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Pre-registration hashed", prereg.fingerprint[:12], True),
    ("All arms and seeds present", ", ".join(missing) or "complete", not missing),
    ("R6 risk maps evaluated", f"{len(risk)} evaluations", not risk.empty),
    ("Verdict on real data", decision["verdict"], decision["verdict"] != "NOT EVALUATED"),
], columns=["item", "detail", "passed"])
display(status)

## What each verdict implies for S6 onwards

| Verdict | Next |
|---|---|
| GAIN | Dosimetry on the 20-case subset (S6) for R1, R3, R4, R5, R6; the γ-proton test completes go/no-go 2. |
| BIAS CORRECTION SUFFICES | The paper becomes "the right estimator suffices": theory (Results A–C), R1 vs R3 vs R4, tails via R6 and the generative arm. |
| NO GAIN | Range loss dropped as a contribution; the paper rests on estimator choice, tails and projected CRPS (Results B–C). |

Whatever the verdict, the generative arm (S7–S8) is compared with R6 on tails and with R3 on the mean, where the theory
predicts equality at best.